# 05 · REVEL — a supervised meta-predictor & the *circularity* problem

**REVEL** (*Rare Exome Variant Ensemble Learner*, Ioannidis et al. 2016, *AJHG*, PMID 27666373) is a **supervised** random-forest **ensemble** of 13 predictors, trained on curated pathogenic/benign variants. That training is powerful — but it makes benchmarking REVEL against ClinVar **partly circular**, which is the key lesson here.

> ✅ **REAL DATA.** Genome-wide **REVEL v1.3** for CFTR — **~10,826** variants (`data/revel_cftr_v1.3.csv`, `build_revel.py`). **Keyed by genomic coordinate** (REVEL has no protein position) — join onto observed variants by `chrom,pos,ref,alt`. **Non-commercial license.** `source == 'REAL'`.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 2 · REVEL — what is it?

**REVEL** = *Rare Exome Variant Ensemble Learner* (Ioannidis et al. 2016, *American Journal of
Human Genetics*, **PMID 27666373**).

Think of REVEL as a **committee vote of 13 other predictors**. Instead of inventing a brand-new way
to judge a missense variant, its authors took the scores of **13 individual tools** (e.g. SIFT,
PolyPhen-2, MutationTaster, and several conservation scores) and trained a **random forest** — a
supervised machine-learning model — to combine them into one number.

Key facts to remember:

| Property | REVEL |
|---|---|
| Learning type | **Supervised** (learned from labelled examples) |
| Model | Random-forest **ensemble** of 13 component predictors |
| Trained on | Curated **pathogenic** and **benign** missense variants |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Covers | Missense variants only |

The "supervised" part is crucial: someone had to **hand REVEL a training set of variants already
labelled pathogenic or benign**. Where did those labels come from? Databases like **ClinVar** and
**HGMD**. Hold that thought — it is the seed of the circularity problem in section 3.


In [2]:
revel = tk.load_revel()    # REAL — genome-wide REVEL v1.3 for CFTR (~10,826), coord-keyed
print(f"{len(revel):,} REAL REVEL variants | source: {revel['source'].unique().tolist()}")
print('columns:', list(revel.columns))
print('revel_score range:', revel['revel_score'].min(), '->', revel['revel_score'].max())
revel.head(6)

10,127 REAL REVEL variants | source: ['REAL']
columns: ['chrom', 'pos', 'ref', 'alt', 'aaref', 'aaalt', 'revel_score', 'source']
revel_score range: 0.002 -> 0.996


,chrom,pos,ref,alt,aaref,aaalt,revel_score,source
0,7,117642459,G,A,G,R,0.996,REAL
1,7,117642459,G,C,G,R,0.996,REAL
2,7,117559510,G,C,G,A,0.995,REAL
3,7,117548822,A,C,K,T,0.995,REAL
4,7,117642514,G,T,G,V,0.995,REAL
5,7,117548813,G,T,G,V,0.995,REAL


## REVEL for the *observed* CFTR variants

REVEL is coordinate-keyed; join it onto the observed gnomAD missense set to attach the protein change and see REVEL over real variants.

In [3]:
gn = tk.load_gnomad_missense()
vid = gn['variant_id'].str.split('-', expand=True)
gn['chrom'], gn['pos'], gn['ref'], gn['alt'] = vid[0], vid[1].astype(int), vid[2], vid[3]
obs = gn.merge(revel[['chrom','pos','ref','alt','revel_score']], on=['chrom','pos','ref','alt'], how='left')
print('observed gnomAD missense with a REAL REVEL score:', int(obs['revel_score'].notna().sum()), '/', len(gn))
obs.dropna(subset=['revel_score']).sort_values('revel_score', ascending=False).head(8)[['protein_variant','hgvs_c','revel_score']]

observed gnomAD missense with a REAL REVEL score: 2436 / 2466


,protein_variant,hgvs_c,revel_score
2073,G1247R,c.3739G>A,0.996
823,K464T,c.1391A>C,0.995
819,G461V,c.1382G>T,0.995
2089,G1265R,c.3793G>A,0.994
822,G463D,c.1388G>A,0.994
2236,G1349D,c.4046G>A,0.994
853,G480D,c.1439G>A,0.993
2075,G1249R,c.3745G>A,0.992


## 3 · ⚠️ THE CIRCULARITY WARNING — the key idea in this notebook

> ### 🔁 Why "REVEL disagrees with ClinVar" can be *misleading*
>
> REVEL was **trained on labels that share lineage with ClinVar and HGMD.**
>
> So when you benchmark REVEL against ClinVar and find they **agree**, that agreement may be
> **partly baked in** — REVEL may simply be *remembering* variants it was trained on. This is
> **label leakage**: information from the "answer key" leaked into the model.
>
> And when REVEL **disagrees** with ClinVar, that disagreement is **not guaranteed to be
> independent evidence** either — it can reflect quirks of the training labels rather than a fresh,
> orthogonal opinion.
>
> **Bottom line:** a supervised predictor benchmarked against its own training-label source is
> **partly circular**. It looks more accurate than it truly is on *new* variants.

**Contrast with the unsupervised tools from notebooks 03-04:**

| Tool | Learned from clinical labels? | Benchmarking vs ClinVar is… |
|---|---|---|
| AlphaMissense, EVE, ESM1b | **No** (sequence / structure / evolution only) | **Fair-ish** — an independent opinion |
| **REVEL** | **Yes** (ClinVar/HGMD lineage) | **Partly circular** — beware label leakage |
| PrimateAI | Partly (a benign *proxy*, not clinical labels) | **Medium** circularity |

➡️ **Forward reference:** Notebook **08** builds the de-circularization / benchmarking analysis
properly — using **orthogonal** truth sets like CFTR2 functional data and population frequency, so
supervised tools are not simply graded against their own homework.


## 4 · REVEL isn't one cut-point — it's a *graded* scale

A very common shortcut is: *"REVEL ≥ 0.75 → likely pathogenic, otherwise not."* That single cut
throws away information.

The ClinGen **Sequence Variant Interpretation** working group (**Pejaver et al. 2022**,
*AJHG*, **PMID 36413997**) *calibrated* REVEL against the **ACMG/AMP** evidence framework. They
showed a REVEL score maps to **graded tiers of pathogenic evidence strength** (Supporting →
Moderate → Strong), not a single yes/no line:

| REVEL score ≥ | ACMG pathogenic evidence strength (approx.) |
|---|---|
| **0.932** | **Strong** (PP3_Strong) |
| **0.773** | **Moderate** (PP3_Moderate) |
| **0.644** | **Supporting** (PP3_Supporting) |
| **0.290** | *below this* → **Benign** supporting evidence (BP4) |

(These break-points are approximate and are the ones used in the toolkit's teaching table.)

**Why it matters:** two variants at REVEL 0.65 and 0.95 are *both* "≥ 0.75-ish territory" under a
single cut, but the calibration says one is only **Supporting** evidence and the other is
**Strong**. Collapsing them to one label loses exactly the nuance a curator needs.

The toolkit deliberately ships a **single** binary cut (`tk.THRESHOLDS['revel']`) to keep the
`call_from_score` helper simple — but you should know the richer, graded reality exists.


In [4]:
# The toolkit's SIMPLE binary cut-points (one 'pathogenic' line, one 'benign' line):
tk.THRESHOLDS['revel']

{'path': 0.75, 'benign': 0.29}

## 6 · How to get the REAL REVEL scores

Download the **genome-wide REVEL table** from `https://sites.google.com/site/revelgenomics/`. It is keyed by **genomic coordinate** (`chr, pos, ref, alt`, GRCh37/38) — join on the coordinate, **not** the protein change (one protein change can arise from several codon changes).

> ⚠️ **CFTR minus-strand gotcha:** make sure ref/alt are on the same genomic strand as the downloaded table, or you silently match nothing (the CADD notebook shows the strand-flip trick).

## 8 · A taste of the graded idea — binning REVEL into the 4 Pejaver tiers

Instead of one line at 0.75, let's sort the DEMO variants into the **four calibrated tiers** from
section 4 using `pandas.cut`. This is a miniature version of what ACMG-style graded evidence looks
like in practice.


In [5]:
tier_edges  = [-np.inf, 0.290, 0.644, 0.773, 0.932, np.inf]
tier_labels = ['Benign-supporting (<0.290)', 'Indeterminate (0.290-0.644)',
               'Path Supporting (0.644-0.773)', 'Path Moderate (0.773-0.932)',
               'Path Strong (>=0.932)']
revel['revel_tier'] = pd.cut(revel['revel_score'], bins=tier_edges, labels=tier_labels, right=False)
print('REAL REVEL CFTR variants per Pejaver 2022 evidence tier:\n')
print(revel['revel_tier'].value_counts().reindex(tier_labels).fillna(0).astype(int).to_string())

REAL REVEL CFTR variants per Pejaver 2022 evidence tier:

revel_tier
Benign-supporting (<0.290)       1049
Indeterminate (0.290-0.644)      4166
Path Supporting (0.644-0.773)    1738
Path Moderate (0.773-0.932)      2359
Path Strong (>=0.932)             815


Even in this tiny DEMO set you can see variants spread across **several** evidence strengths — the
detail a single 0.75 cut-point would flatten into just "pathogenic vs not".


## 7 · What the toolkit records about REVEL

`toolkit.py` keeps a registry entry per predictor — its learning type and **circularity** rating — which notebook 12 uses to decide which tools can be fairly benchmarked.

In [6]:
info = tk.TOOL_REGISTRY['REVEL']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : supervised
  signal      : random-forest ENSEMBLE of 13 scores, trained on curated pathogenic/benign labels
  circularity : HIGH (label lineage overlaps ClinVar/HGMD)
  pmid        : 27666373


## The shared A1 worked-example panel — **REVEL** in context

The same ~14 famous CFTR **missense** variants are scored by *every* A1 tool
(notebooks 01–08), so you can follow one fixed panel across the whole series. Your
column here is **`REVEL`**; the CFTR2 / ClinVar truth columns are shown for context. REVEL is coordinate-keyed, so it is bridged onto each variant's gnomAD genomic coordinate — the join-key lesson in action.

`tk.a1_panel()` is the single source of truth (defined in `toolkit.py`); it uses the
REAL extracts where present (missing extracts → NaN).

In [7]:
tk.a1_panel()

,protein_variant,gnomad_af,AM,EVE,ESM1b,REVEL,PAI,cftr2_class,clinvar_call
0,G551D,0.000404,0.9897,0.939098,-12.123,0.990,0.757039,CF-causing,pathogenic
1,F508del,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,R117H,0.002199,0.2985,0.819590,-5.659,0.807,0.538583,Varying clinical consequence,pathogenic
3,R334W,0.000092,0.6563,0.950636,-8.179,0.816,0.715262,CF-causing,pathogenic
4,G85E,0.000065,0.9881,0.922779,-11.889,0.940,0.775947,CF-causing,pathogenic
5,D1152H,0.000407,0.8597,0.842551,-9.784,0.657,0.687562,Varying clinical consequence,uncertain
6,R668C,0.008450,0.1040,0.921812,-9.968,0.706,0.713809,Non CF-causing,uncertain
7,Y161C,0.000002,0.8006,0.934065,-8.857,0.969,0.844116,No interpretation available,uncertain
8,G970D,0.000007,0.7638,0.930015,-12.000,0.985,0.621215,CF-causing,pathogenic
9,S912L,0.000663,0.1245,0.085494,-3.418,0.543,0.301304,Varying clinical consequence,uncertain


## Key takeaways

1. **REVEL** is a **supervised** ensemble; score 0-1, higher = worse. Now **REAL** — genome-wide REVEL v1.3 for CFTR (~10,826), coordinate-keyed.
2. ⚠️ **Circularity stays:** REVEL's labels share lineage with ClinVar/HGMD, so grading it against ClinVar is **partly circular** — don't treat REVEL-vs-ClinVar agreement as independent (notebook 12).
3. Read REVEL as a **graded** scale (Pejaver 2022), not a single 0.75 cut.
4. Non-commercial license — cite REVEL; raw table kept external.

**Next:** notebook 06 — **PrimateAI**.